In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

from scipy.ndimage import gaussian_filter
import re

from pybedtools import BedTool
from sklearn.cluster import KMeans

from matplotlib_venn import venn2

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai',  'Delayed', 'Fail1','Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

plot_order = ['Start', 'Delayed', 'Fail1','Sendai', 'Inter1',  'Inter2', 'Fail2', 'IPS']

In [ ]:
genes_path = '/u/project/cluo/terencew/reference/hg38_igvf/bed/genes/gencode.v43.autosome.dedup.names.bed'
genes = pd.read_csv(genes_path, sep='\t', header=None)
genes.index = [f'{x}_{y}' for x,y in zip(genes[0], genes[1])]
genes['chr_num'] = [int(x.split('chr')[1]) for x in genes[0]]
genes = genes.sort_values(by=['chr_num', 1]).iloc[:,:4]
genes.shape

### load enhancer methylation

In [ ]:
peaks_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/atac/peaks/chromhmm/justin.chromhmm.dedup.bed.gz'
peaks = pd.read_csv(peaks_path, sep='\t', header=None)
peaks.index = [f'{x}_{y}' for x,y in zip(peaks[0], peaks[1])]
peaks.shape

In [ ]:
peaks.head()

In [ ]:
peaks_df = pd.read_csv(f'{projdir}/csv/peaks/enhancers/enhancer_mcg.csv', sep='\t', index_col=0)
peaks_df.shape

In [ ]:
peaks_df.head()

In [ ]:
a = 'Enhancer'
annot_peaks = peaks[peaks[14] == a]
annot_peaks = annot_peaks[~annot_peaks.index.duplicated()]
annot_peaks.shape

In [ ]:
annot_peaks.head()

In [ ]:
enh_df = peaks_df.reindex(annot_peaks.index)
enh_df.shape

In [ ]:
prod_traj = ['Start', 'Sendai', 'Inter1', 'Inter2', 'IPS'] 

s1 = prod_traj[0]
prod_diff = pd.DataFrame(index=enh_df.index)
for s2 in prod_traj[1:]:
    diff = (peaks_df[s2] - peaks_df[s1])
    tmp_str = f'{s1}_{s2}'
    prod_diff[tmp_str] = diff
    s1 = s2
    
prod_diff['Start_Inter1'] = peaks_df['Inter1'] - peaks_df['Start']
prod_diff.head()

In [ ]:
fail_traj = ['Start', 'Delayed', 'Fail1']

s1 = fail_traj[0]
fail_diff = pd.DataFrame(index=enh_df.index)
for s2 in fail_traj[1:]:
    diff = (peaks_df[s2] - peaks_df[s1])
    tmp_str = f'{s1}_{s2}'
    fail_diff[tmp_str] = diff
    s1 = s2
fail_diff.head()

### plot histogram between start_sendai and start_delayed

In [ ]:
fig, ax = plt.subplots(1, figsize=(4,4))
sns.histplot(prod_diff['Start_Sendai'])

In [ ]:
fig, ax = plt.subplots(1, figsize=(4,4))
sns.histplot(prod_diff['Start_Inter1'])

In [ ]:
fig, ax = plt.subplots(1, figsize=(4,4))
sns.histplot(fail_diff['Start_Delayed'])

In [ ]:
select = ['Start_Sendai']
tmp_prod = prod_diff[select].copy()
tmp_prod['type'] = 'Start_Sendai'
tmp_prod.columns = ['mCG', 'type']

select = ['Start_Delayed']
tmp_fail = fail_diff[select].copy()
tmp_fail['type'] = 'Start_Delayed'
tmp_fail.columns = ['mCG', 'type']

merged = pd.concat([tmp_prod, tmp_fail], axis=0)
merged.shape

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
merged.to_csv(f'{outdir}/start_sendai_enhancers_mcg_diff.csv', sep='\t')

In [ ]:
select = ['Start_Inter1']
tmp_prod = prod_diff[select].copy()
tmp_prod['type'] = 'Start_Inter1'
tmp_prod.columns = ['mCG', 'type']
          
select = ['Start_Delayed']
tmp_fail = fail_diff[select].copy()
tmp_fail['type'] = 'Start_Delayed'
tmp_fail.columns = ['mCG', 'type']

merged = pd.concat([tmp_prod, tmp_fail], axis=0)
merged.shape

In [ ]:
merged.head()

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
merged.to_csv(f'{outdir}/start_inter1_sendai_enhancers_mcg_diff.csv', sep='\t')

### read enhancers


In [ ]:
prod_pos_enh = pd.read_csv(f'{projdir}/csv/peaks/enhancers/prod_pos_enh.csv', sep='\t', index_col=0)
prod_neg_enh = pd.read_csv(f'{projdir}/csv/peaks/enhancers/prod_neg_enh.csv', sep='\t', index_col=0)
fail_pos_enh = pd.read_csv(f'{projdir}/csv/peaks/enhancers/fail_pos_enh.csv', sep='\t', index_col=0)
fail_neg_enh = pd.read_csv(f'{projdir}/csv/peaks/enhancers/fail_neg_enh.csv', sep='\t', index_col=0)

In [ ]:
prod_pos_enh.shape, prod_neg_enh.shape, \
fail_pos_enh.shape, fail_neg_enh.shape

In [ ]:
prod_pos_enh.head()

In [ ]:
k = 6
kmeans = KMeans(n_clusters=k)
kmeans.fit(prod_neg_enh)
k_clusters = kmeans.predict(prod_neg_enh)
prod_neg_enh['cluster'] = k_clusters
cluster_sizes = prod_neg_enh['cluster'].value_counts().sort_index().values
prod_neg_enh = prod_neg_enh.sort_values(by=['cluster']).iloc[:,:4]

prod_neg_boundaries = np.cumsum(cluster_sizes)[:-1]

ax = sns.heatmap(prod_neg_enh, cmap='coolwarm')

for b in prod_neg_boundaries:
    ax.hlines(b, *ax.get_xlim(), colors='black', linewidth=2)

plt.show()

In [ ]:
k = 6
kmeans = KMeans(n_clusters=k)
kmeans.fit(fail_pos_enh)
k_clusters = kmeans.predict(fail_pos_enh)
fail_pos_enh['cluster'] = k_clusters
cluster_sizes = fail_pos_enh['cluster'].value_counts().sort_index().values
fail_pos_enh = fail_pos_enh.sort_values(by=['cluster']).iloc[:,:2]

fail_pos_boundaries = np.cumsum(cluster_sizes)[:-1]

ax = sns.heatmap(fail_pos_enh, cmap='coolwarm')

for b in fail_pos_boundaries:
    ax.hlines(b, *ax.get_xlim(), colors='black', linewidth=2)

plt.show()

In [ ]:
k = 6
kmeans = KMeans(n_clusters=k)
kmeans.fit(fail_neg_enh)
k_clusters = kmeans.predict(fail_neg_enh)
fail_neg_enh['cluster'] = k_clusters
cluster_sizes = fail_neg_enh['cluster'].value_counts().sort_index().values
fail_neg_enh = fail_neg_enh.sort_values(by=['cluster']).iloc[:,:2]

fail_neg_boundaries = np.cumsum(cluster_sizes)[:-1]

ax = sns.heatmap(fail_neg_enh, cmap='coolwarm')

for b in fail_neg_boundaries:
    ax.hlines(b, *ax.get_xlim(), colors='black', linewidth=2)

plt.show()

### just k-means the og

In [ ]:
# ###

In [ ]:
final_prod_neg = peaks_df.loc[prod_neg_enh.index][plot_order].dropna()
final_fail_neg = peaks_df.loc[fail_neg_enh.index][plot_order].dropna()
final_prod_pos = peaks_df.loc[prod_pos_enh.index][plot_order].dropna()
final_fail_pos = peaks_df.loc[fail_pos_enh.index][plot_order].dropna()

In [ ]:
k = 6
kmeans = KMeans(n_clusters=k)
kmeans.fit(final_prod_pos)
k_clusters = kmeans.predict(final_prod_pos)
final_prod_pos['cluster'] = k_clusters
cluster_sizes = final_prod_pos['cluster'].value_counts().sort_index().values
final_prod_pos = final_prod_pos.sort_values(by=['cluster']).iloc[:,:-1]

prod_pos_boundaries = np.cumsum(cluster_sizes)[:-1]
print(prod_pos_boundaries)

ax = sns.heatmap(final_prod_pos, cmap='coolwarm')

for b in prod_pos_boundaries:
    ax.hlines(b, *ax.get_xlim(), colors='black', linewidth=2)

plt.show()

In [ ]:
k = 6
kmeans = KMeans(n_clusters=k)
kmeans.fit(final_prod_neg)
k_clusters = kmeans.predict(final_prod_neg)
final_prod_neg['cluster'] = k_clusters
cluster_sizes = final_prod_neg['cluster'].value_counts().sort_index().values
final_prod_neg = final_prod_neg.sort_values(by=['cluster']).iloc[:,:-1]

prod_neg_boundaries = np.cumsum(cluster_sizes)[:-1]
print(prod_neg_boundaries)

ax = sns.heatmap(final_prod_neg, cmap='coolwarm')

for b in prod_neg_boundaries:
    ax.hlines(b, *ax.get_xlim(), colors='black', linewidth=2)

plt.show()

In [ ]:
k = 6
kmeans = KMeans(n_clusters=k)
kmeans.fit(final_fail_pos)
k_clusters = kmeans.predict(final_fail_pos)
final_fail_pos['cluster'] = k_clusters
cluster_sizes = final_fail_pos['cluster'].value_counts().sort_index().values
final_fail_pos = final_fail_pos.sort_values(by=['cluster']).iloc[:,:-1]

fail_pos_boundaries = np.cumsum(cluster_sizes)[:-1]
print(fail_pos_boundaries)

ax = sns.heatmap(final_fail_pos, cmap='coolwarm')

for b in fail_pos_boundaries:
    ax.hlines(b, *ax.get_xlim(), colors='black', linewidth=2)

plt.show()

In [ ]:
k = 6
kmeans = KMeans(n_clusters=k)
kmeans.fit(final_fail_neg)
k_clusters = kmeans.predict(final_fail_neg)
final_fail_neg['cluster'] = k_clusters
cluster_sizes = final_fail_neg['cluster'].value_counts().sort_index().values
final_fail_neg = final_fail_neg.sort_values(by=['cluster']).iloc[:,:-1]

fail_neg_boundaries = np.cumsum(cluster_sizes)[:-1]
print(fail_neg_boundaries)

ax = sns.heatmap(final_fail_neg, cmap='coolwarm')

for b in fail_neg_boundaries:
    ax.hlines(b, *ax.get_xlim(), colors='black', linewidth=2)

plt.show()

### how much do prod/fail neg enhancers overlap?

In [ ]:
prod_neg_enh.index.isin(fail_neg_enh.index).sum(), \
prod_neg_enh.index.isin(fail_neg_enh.index).sum() / prod_neg_enh.shape[0]

In [ ]:
plt.subplots(1, figsize=(4,4))
prod_pos_set = set(prod_pos_enh.index)
fail_pos_set = set(fail_pos_enh.index)

to_plot = [prod_pos_set, fail_pos_set]
ax = venn2(to_plot)

In [ ]:
plt.subplots(1, figsize=(4,4))
prod_neg_set = set(prod_neg_enh.index)
fail_neg_set = set(fail_neg_enh.index)

to_plot = [prod_neg_set, fail_neg_set]
ax = venn2(to_plot)

### write to final figures

In [ ]:
vals = [prod_neg_enh.shape[0], prod_pos_enh.shape[0],
        fail_neg_enh.shape[0], fail_pos_enh.shape[0]]
count_df = pd.DataFrame(vals)
count_df.columns = ['num_enhancers']
count_df['branch'] = np.repeat(['Productive', 'Failure'], 2)
count_df['mcg_change'] = np.tile(['negative', 'positive'], 2)
count_df.head()

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
count_df.to_csv(f'{outdir}/sendai_enhancers_delta_mcg.csv', sep='\t')

In [ ]:
# tmp_merged.shape

In [ ]:
### I like this way for now
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
final_prod_neg.to_csv(f'{outdir}/sendai_enhancers_prod_neg.csv', sep='\t')
final_fail_neg.to_csv(f'{outdir}/sendai_enhancers_fail_neg.csv', sep='\t')
final_fail_pos.to_csv(f'{outdir}/sendai_enhancers_fail_pos.csv', sep='\t')

### venn

In [ ]:
prod_neg_df = pd.DataFrame(prod_neg_set)
prod_neg_df['branch'] = 'prod'
fail_neg_df = pd.DataFrame(fail_neg_set)
fail_neg_df['branch'] = 'fail'

merged_neg = pd.concat([prod_neg_df, fail_neg_df])
merged_neg.columns = ['enhancer', 'branch']
merged_neg.head()

In [ ]:
prod_pos_df = pd.DataFrame(prod_pos_set)
prod_pos_df['branch'] = 'prod'
fail_pos_df = pd.DataFrame(fail_pos_set)
fail_pos_df['branch'] = 'fail'

merged_pos = pd.concat([prod_pos_df, fail_pos_df])
merged_pos.columns = ['enhancer', 'branch']
merged_pos.head()

In [ ]:
merged_pos['type'] = 'pos'
merged_neg['type'] = 'neg'
final_venn = pd.concat([merged_pos, merged_neg])
final_venn.index.name = 'index'

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
final_venn.to_csv(f'{outdir}/sendai_enhancers_venn.csv', sep='\t')

### methylation box plot delta

In [ ]:
prod_neg = prod_diff.loc[fail_neg_enh.index]
fail_neg = fail_diff.loc[fail_neg_enh.index]
prod_neg.shape, fail_neg.shape

In [ ]:
prod_neg = prod_diff.loc[fail_neg_enh.index]
fail_neg = fail_diff.loc[fail_neg_enh.index]
prod_plot = prod_neg.melt(value_vars=prod_neg.columns)
prod_plot['branch'] = 'Productive'
fail_plot = fail_neg.melt(value_vars=fail_neg.columns)
fail_plot['branch'] = 'Failure'
fail_neg_plot = pd.concat([fail_plot, prod_plot]).reset_index(drop=True)
sns.boxplot(data=fail_neg_plot, x='variable', y='value', hue='branch', showfliers=False)

### prod neg

In [ ]:
prod_neg = prod_diff.loc[prod_neg_enh.index]
fail_neg = fail_diff.loc[prod_neg_enh.index]
prod_plot = prod_neg.melt(value_vars=prod_neg.columns)
prod_plot['branch'] = 'Productive'
fail_plot = fail_neg.melt(value_vars=fail_neg.columns)
fail_plot['branch'] = 'Failure'
prod_neg_plot = pd.concat([fail_plot, prod_plot]).reset_index(drop=True)
sns.boxplot(data=prod_neg_plot, x='variable', y='value', hue='branch', showfliers=False)

In [ ]:
prod_neg_plot['type'] = 'Productive -mCG enhancers'
fail_neg_plot['type'] = 'Fail1 -mCG enhancers'

final_plot = pd.concat([prod_neg_plot, fail_neg_plot])
final_plot.head()

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
final_plot.to_csv(f'{outdir}/sendai_enhancers_delta_reversal.csv', sep='\t')

### scatter plot

In [ ]:
!date